# ROTIR Image Reconstruction Demo

**Complete interferometric imaging pipeline for stellar surface mapping**

This notebook demonstrates:
1. Loading real OIFITS data (CHARA, VLTI, etc.)
2. Setting up stellar models (single stars, binaries, rapid rotators)
3. Image reconstruction with regularization
4. Visualization and analysis

---

## Applications:
- **Red Supergiants:** Convection cells, limb darkening
- **Symbiotic Stars:** Binary systems, Roche geometry, mass transfer
- **Rapid Rotators:** Gravity darkening (Altair, Vega)
- **Spotted Stars:** Starspots, magnetic activity

---

In [ ]:
# Imports
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from pathlib import Path

# ROTIR modules
import sys
sys.path.insert(0, 'rotir_jax')

from rotir_jax.io.oifits_reader import read_oifits
from rotir_jax.tessellation.healpix import tessellation_healpix
from rotir_jax.geometry.base import create_star
from rotir_jax.geometry.rapid_rotator import create_rapid_rotator_star
from rotir_jax.geometry.orbits import binary_orbit_absolute
from rotir_jax.geometry.roche import eggleton_roche_radius
from rotir_jax.reconstruction.optimizer import (
    StellarImageReconstructor,
    reconstruct_stellar_surface,
    compute_reduced_chi2,
)
from rotir_jax.reconstruction.multi_epoch import (
    Epoch,
    reconstruct_multi_epoch,
    compute_rotation_phase,
)

print("✓ Imports successful")

---

# Part 1: Single Red Supergiant

Red supergiants are ideal targets for interferometry:
- Large angular size (2-50 mas)
- Convection cells (~0.1-0.5 stellar radius)
- Strong limb darkening
- Time-variable surface structure

Examples: Betelgeuse, Antares, Aldebaran, μ Cep

In [ ]:
# ========================================
# Configuration: Red Supergiant
# ========================================

# Path to your OIFITS file
OIFITS_FILE = "path/to/your/red_supergiant.fits"  # <-- EDIT THIS

# Stellar parameters
DIAMETER = 44.0  # mas (e.g., Betelgeuse)
T_EFF = 3500.0   # K (effective temperature)
INCLINATION = 90.0  # degrees (pole-on = 0, edge-on = 90)
ORIENTATION = 0.0   # degrees (rotation angle)

# Reconstruction parameters
NSIDE = 4  # HEALPix resolution (4 → 192 pixels, 8 → 768 pixels)
T_MIN = 2500.0  # K (temperature bounds)
T_MAX = 4500.0  # K
MAXITER = 200   # Optimization iterations

print(f"Target: Red Supergiant")
print(f"  Diameter: {DIAMETER} mas")
print(f"  T_eff: {T_EFF} K")
print(f"  Resolution: {12 * NSIDE**2} pixels")

In [ ]:
# Load OIFITS data
print(f"Loading: {OIFITS_FILE}")
oi_data = read_oifits(OIFITS_FILE)

print(f"\n✓ Data loaded:")
print(f"  Wavelengths: {len(oi_data.wavelengths)} ({oi_data.wavelengths.min()*1e6:.2f}-{oi_data.wavelengths.max()*1e6:.2f} μm)")
print(f"  Vis²: {len(oi_data.vis2)} measurements")
print(f"  Closure phases: {len(oi_data.t3phi)} measurements")
print(f"  Baselines: {oi_data.u.min():.1f} to {oi_data.u.max():.1f} m")

In [ ]:
# Create stellar model
tess = tessellation_healpix(n=NSIDE)

# Initial guess: uniform photosphere
initial_temp = jnp.ones(tess.npix) * T_EFF

star = create_star(
    tess=tess,
    inclination=INCLINATION,
    orientation=ORIENTATION,
    intensities=initial_temp / T_EFF,  # Normalized
    diameter=DIAMETER,
)

print(f"\n✓ Star model created:")
print(f"  Pixels: {tess.npix}")
print(f"  Visible: {jnp.sum(star.visible)} pixels")
print(f"  Diameter: {DIAMETER} mas")

In [ ]:
# Define regularizers for red supergiant
# - MEM: Smooth baseline (convection cells are large-scale)
# - TV: Preserve cell boundaries

regularizers_rsg = [
    {"type": "mem", "weight": 0.05},   # Maximum entropy (smoothness)
    {"type": "tv", "weight": 0.01},    # Total variation (edges)
]

print("Regularizers:")
for reg in regularizers_rsg:
    print(f"  {reg['type'].upper()}: λ = {reg['weight']}")

In [ ]:
# RECONSTRUCT!
print("\n" + "="*80)
print("STARTING RECONSTRUCTION")
print("="*80)

result_rsg = reconstruct_stellar_surface(
    oi_data=oi_data,
    star=star,
    x_start=initial_temp,
    regularizers=regularizers_rsg,
    bounds=(T_MIN, T_MAX),
    maxiter=MAXITER,
    verbose=True,
)

print("\n" + "="*80)
print("RECONSTRUCTION COMPLETE")
print("="*80)
print(f"Success: {result_rsg.success}")
print(f"Iterations: {result_rsg.iterations}")
print(f"Final χ²: {result_rsg.chi2_final:.2f}")
print(f"χ²_red: {result_rsg.chi2_final / (len(oi_data.vis2) + len(oi_data.t3phi)):.2f}")

In [ ]:
# Extract temperature map
temperature_map_rsg = result_rsg.x_solution

print("\nReconstructed Temperature Map:")
print(f"  Min: {temperature_map_rsg.min():.0f} K")
print(f"  Max: {temperature_map_rsg.max():.0f} K")
print(f"  Mean: {temperature_map_rsg.mean():.0f} K")
print(f"  Contrast: {(temperature_map_rsg.max() - temperature_map_rsg.min()):.0f} K ({100*(temperature_map_rsg.max() - temperature_map_rsg.min())/temperature_map_rsg.mean():.1f}%)")

In [ ]:
# Visualize convergence
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Objective function
ax = axes[0]
ax.plot(result_rsg.history['f_total'], 'k-', label='Total', linewidth=2)
ax.plot(result_rsg.history['chi2'], 'r-', label='χ²', alpha=0.7)
ax.plot(result_rsg.history['reg'], 'b-', label='Regularization', alpha=0.7)
ax.set_xlabel('Function Evaluation')
ax.set_ylabel('Objective Value')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('Convergence')

# Temperature statistics
ax = axes[1]
ax.plot(result_rsg.history['x_max'], 'r-', label='Max', linewidth=2)
ax.plot(result_rsg.history['x_mean'], 'k-', label='Mean', linewidth=2)
ax.plot(result_rsg.history['x_min'], 'b-', label='Min', linewidth=2)
ax.set_xlabel('Function Evaluation')
ax.set_ylabel('Temperature (K)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('Temperature Evolution')

# Temperature histogram
ax = axes[2]
ax.hist(temperature_map_rsg, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(T_EFF, color='r', linestyle='--', label=f'T_eff = {T_EFF} K')
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Number of Pixels')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('Temperature Distribution')

plt.tight_layout()
plt.savefig('reconstruction_convergence_rsg.png', dpi=150)
print("\n✓ Saved: reconstruction_convergence_rsg.png")
plt.show()

In [ ]:
# Visualize surface map (Mollweide projection)
fig, ax = plt.subplots(1, 1, figsize=(12, 6), subplot_kw={'projection': 'mollweide'})

# Convert to Mollweide coordinates
lon = tess.phi - np.pi  # [-π, π]
lat = np.pi/2 - tess.theta  # [-π/2, π/2]

# Plot
sc = ax.scatter(lon, lat, c=temperature_map_rsg, s=200, cmap='hot', 
                vmin=T_MIN, vmax=T_MAX, edgecolors='none')

# Colorbar
cbar = plt.colorbar(sc, ax=ax, orientation='horizontal', pad=0.05, fraction=0.046)
cbar.set_label('Temperature (K)', fontsize=12)

ax.set_title(f'Reconstructed Surface Map - Red Supergiant\n(χ²_red = {result_rsg.chi2_final / (len(oi_data.vis2) + len(oi_data.t3phi)):.2f})', 
             fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('surface_map_rsg.png', dpi=150)
print("✓ Saved: surface_map_rsg.png")
plt.show()

---

# Part 2: Symbiotic Star (Binary System)

Symbiotic stars are interacting binaries:
- Red giant + white dwarf
- Roche lobe geometry (tidal distortion)
- Mass transfer through L1 point
- Orbital motion

Examples: R Aquarii, Mira AB, CH Cyg

In [ ]:
# ========================================
# Configuration: Symbiotic Star
# ========================================

# Path to OIFITS file
OIFITS_FILE_SYM = "path/to/your/symbiotic_star.fits"  # <-- EDIT THIS

# Binary parameters
SEPARATION = 100.0  # mas (semi-major axis)
MASS_RATIO = 0.5    # M2/M1 (WD/giant)
ECCENTRICITY = 0.2  # Orbital eccentricity
PERIOD = 640.0      # days (orbital period)
INCLINATION_BIN = 60.0  # degrees

# Giant star parameters
R_GIANT = 40.0  # mas (polar radius)
T_GIANT = 3000.0  # K

# Reconstruction
NSIDE_BIN = 3  # Lower resolution for binaries (48 pixels)

print(f"Target: Symbiotic Star (Binary)")
print(f"  Separation: {SEPARATION} mas")
print(f"  Mass ratio q: {MASS_RATIO}")
print(f"  Period: {PERIOD} days")
print(f"  Giant radius: {R_GIANT} mas")

In [ ]:
# Compute Roche lobe size
R_roche = eggleton_roche_radius(MASS_RATIO) * SEPARATION

print(f"\nRoche Lobe Analysis:")
print(f"  Roche radius: {R_roche:.1f} mas")
print(f"  Giant radius: {R_GIANT:.1f} mas")
print(f"  Fillout factor: {R_GIANT / R_roche:.2f}")

if R_GIANT / R_roche > 0.95:
    print(f"  ⚠️  ROCHE LOBE OVERFLOW - Mass transfer likely!")
elif R_GIANT / R_roche > 0.8:
    print(f"  ⚠️  Near Roche lobe - tidal distortion expected")
else:
    print(f"  ✓ Detached - minimal tidal distortion")

In [ ]:
# Compute binary orbit at observation epoch
EPOCH_MJD = 59000.0  # <-- EDIT: Your observation MJD
T0_MJD = 58000.0     # <-- EDIT: Time of periastron

x1, y1, z1, x2, y2, z2 = binary_orbit_absolute(
    a=SEPARATION,
    e=ECCENTRICITY,
    P=PERIOD,
    T0=T0_MJD,
    q=MASS_RATIO,
    Omega=0.0,  # deg
    i=INCLINATION_BIN,  # deg
    omega=0.0,  # deg
    tepoch=EPOCH_MJD,
)

print(f"\nOrbital Positions at MJD {EPOCH_MJD}:")
print(f"  Giant (primary): ({x1:.2f}, {y1:.2f}, {z1:.2f}) mas")
print(f"  WD (secondary): ({x2:.2f}, {y2:.2f}, {z2:.2f}) mas")
print(f"  Projected separation: {np.sqrt((x2-x1)**2 + (y2-y1)**2):.2f} mas")

In [ ]:
# Load symbiotic star data
print(f"Loading: {OIFITS_FILE_SYM}")
oi_data_sym = read_oifits(OIFITS_FILE_SYM)

print(f"\n✓ Data loaded:")
print(f"  Vis²: {len(oi_data_sym.vis2)}")
print(f"  Closure phases: {len(oi_data_sym.t3phi)}")

In [ ]:
# Create giant star model (focused on giant, WD too small to resolve)
tess_sym = tessellation_healpix(n=NSIDE_BIN)

initial_temp_sym = jnp.ones(tess_sym.npix) * T_GIANT

star_giant = create_star(
    tess=tess_sym,
    inclination=INCLINATION_BIN,
    orientation=0.0,
    intensities=initial_temp_sym / T_GIANT,
    diameter=2*R_GIANT,  # Diameter from radius
)

# Note: For full binary reconstruction, would need to model both components
# Here we focus on the giant (WD is point source)

print(f"\n✓ Giant star model created:")
print(f"  Pixels: {tess_sym.npix}")
print(f"  Diameter: {2*R_GIANT} mas")

In [ ]:
# Regularizers for symbiotic star
# - Less aggressive (larger structures expected)
# - May have spots from magnetic activity

regularizers_sym = [
    {"type": "mem", "weight": 0.03},   # Lighter smoothness
    {"type": "tv", "weight": 0.005},   # Lighter edge preservation
]

print("Regularizers for symbiotic star:")
for reg in regularizers_sym:
    print(f"  {reg['type'].upper()}: λ = {reg['weight']}")

In [ ]:
# RECONSTRUCT SYMBIOTIC STAR
print("\n" + "="*80)
print("RECONSTRUCTING SYMBIOTIC STAR (GIANT COMPONENT)")
print("="*80)

result_sym = reconstruct_stellar_surface(
    oi_data=oi_data_sym,
    star=star_giant,
    x_start=initial_temp_sym,
    regularizers=regularizers_sym,
    bounds=(2000.0, 4000.0),  # K (cooler giant)
    maxiter=150,
    verbose=True,
)

print("\n" + "="*80)
print(f"Success: {result_sym.success}")
print(f"Final χ²_red: {result_sym.chi2_final / (len(oi_data_sym.vis2) + len(oi_data_sym.t3phi)):.2f}")

In [ ]:
# Visualize symbiotic star map
temperature_map_sym = result_sym.x_solution

fig, ax = plt.subplots(1, 1, figsize=(12, 6), subplot_kw={'projection': 'mollweide'})

lon = tess_sym.phi - np.pi
lat = np.pi/2 - tess_sym.theta

sc = ax.scatter(lon, lat, c=temperature_map_sym, s=300, cmap='hot', 
                vmin=2000, vmax=4000, edgecolors='none')

cbar = plt.colorbar(sc, ax=ax, orientation='horizontal', pad=0.05, fraction=0.046)
cbar.set_label('Temperature (K)', fontsize=12)

ax.set_title(f'Symbiotic Star - Giant Component\n(χ²_red = {result_sym.chi2_final / (len(oi_data_sym.vis2) + len(oi_data_sym.t3phi)):.2f})', 
             fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('surface_map_symbiotic.png', dpi=150)
print("✓ Saved: surface_map_symbiotic.png")
plt.show()

---

# Part 3: Multi-Epoch Reconstruction

If you have observations at multiple epochs, ROTIR can reconstruct a single map from all phases (rotation mapping) or track evolution.

In [ ]:
# ========================================
# Multi-Epoch Example (if applicable)
# ========================================

# If you have multiple OIFITS files at different epochs:
OIFITS_FILES = [
    "path/to/epoch1.fits",
    "path/to/epoch2.fits",
    "path/to/epoch3.fits",
    # Add more...
]

# Rotation parameters
ROTATION_PERIOD = 25.0  # days
MJD_REF = 59000.0       # Reference epoch

# Load all epochs
epochs = []
for i, filename in enumerate(OIFITS_FILES):
    # Load data
    data = read_oifits(filename)
    
    # Compute rotation phase
    mjd = 59000.0 + i * 5.0  # Example: 5 days apart
    phase = compute_rotation_phase(mjd, MJD_REF, ROTATION_PERIOD)
    
    epochs.append(Epoch(
        oi_data=data,
        rotation_phase=phase,
        mjd=mjd,
    ))
    
    print(f"Epoch {i+1}: MJD={mjd:.1f}, phase={phase:.3f}")

# Reconstruct (static mode: single map, different viewing angles)
if len(epochs) > 1:
    print("\n" + "="*80)
    print("MULTI-EPOCH RECONSTRUCTION (Static Mode)")
    print("="*80)
    
    result_multi = reconstruct_multi_epoch(
        epochs=epochs,
        star=star,
        mode="static",  # Single map
        regularizers=regularizers_rsg,
        maxiter=200,
        verbose=True,
    )
    
    print(f"\nMulti-epoch success: {result_multi.success}")
    temperature_map_multi = result_multi.x_solution
else:
    print("\n⚠️  Multi-epoch requires multiple OIFITS files")
    print("   Edit OIFITS_FILES list above with your data")

---

# Part 4: Analysis & Export

Export results for further analysis

In [ ]:
# Save reconstructed maps
np.save('temperature_map_rsg.npy', np.array(temperature_map_rsg))
np.save('tessellation_theta.npy', np.array(tess.theta))
np.save('tessellation_phi.npy', np.array(tess.phi))

print("✓ Saved temperature maps:")
print("  - temperature_map_rsg.npy")
print("  - tessellation_theta.npy")
print("  - tessellation_phi.npy")

In [ ]:
# Compute statistics
def compute_statistics(T_map, T_ref):
    """Compute surface statistics."""
    T_min = T_map.min()
    T_max = T_map.max()
    T_mean = T_map.mean()
    T_std = T_map.std()
    contrast = (T_max - T_min) / T_mean
    
    print(f"Temperature Statistics:")
    print(f"  Min: {T_min:.0f} K")
    print(f"  Max: {T_max:.0f} K")
    print(f"  Mean: {T_mean:.0f} K (ref: {T_ref:.0f} K)")
    print(f"  Std: {T_std:.0f} K")
    print(f"  Contrast: {100*contrast:.1f}%")
    print(f"  RMS: {np.sqrt(np.mean((T_map - T_mean)**2)):.0f} K")
    
print("\nRed Supergiant:")
compute_statistics(temperature_map_rsg, T_EFF)

In [ ]:
# Export reconstruction report
with open('reconstruction_report.txt', 'w') as f:
    f.write("ROTIR Reconstruction Report\n")
    f.write("="*60 + "\n\n")
    
    f.write(f"Red Supergiant Reconstruction\n")
    f.write(f"-" * 40 + "\n")
    f.write(f"Data file: {OIFITS_FILE}\n")
    f.write(f"Vis²: {len(oi_data.vis2)}\n")
    f.write(f"Closure phases: {len(oi_data.t3phi)}\n")
    f.write(f"\nModel:\n")
    f.write(f"  Pixels: {tess.npix}\n")
    f.write(f"  Diameter: {DIAMETER} mas\n")
    f.write(f"  T_eff: {T_EFF} K\n")
    f.write(f"\nReconstruction:\n")
    f.write(f"  Success: {result_rsg.success}\n")
    f.write(f"  Iterations: {result_rsg.iterations}\n")
    f.write(f"  χ²: {result_rsg.chi2_final:.2f}\n")
    f.write(f"  χ²_red: {result_rsg.chi2_final / (len(oi_data.vis2) + len(oi_data.t3phi)):.2f}\n")
    f.write(f"\nTemperature Map:\n")
    f.write(f"  Min: {temperature_map_rsg.min():.0f} K\n")
    f.write(f"  Max: {temperature_map_rsg.max():.0f} K\n")
    f.write(f"  Mean: {temperature_map_rsg.mean():.0f} K\n")
    f.write(f"  Contrast: {100*(temperature_map_rsg.max() - temperature_map_rsg.min())/temperature_map_rsg.mean():.1f}%\n")

print("\n✓ Saved: reconstruction_report.txt")

---

# Summary

This notebook demonstrated:
- ✅ Loading real OIFITS data
- ✅ Red supergiant reconstruction (convection cells)
- ✅ Symbiotic star reconstruction (binary system)
- ✅ Multi-epoch reconstruction (optional)
- ✅ Visualization and analysis

## Next Steps:
1. **Adjust regularization weights** for better fits
2. **Try different resolutions** (NSIDE = 2, 4, 8)
3. **Compare with models** (uniform disk, limb-darkened, etc.)
4. **Multi-wavelength** reconstruction
5. **Publish results!** 📝

---

**From photons to pixels: Mapping the surfaces of distant stars** ✨